#Projet SPARK

##2. Mise en place de l’environnement de travail

###2. Installation de Spark comme vu en cours

In [54]:
! apt-get install openjdk-8-jdk-headless -qq > /dev/null
! wget -q https://dlcdn.apache.org/spark/spark-3.5.0/spark-3.5.0-bin-hadoop3.tgz
! tar xf /content/spark-3.5.0-bin-hadoop3.tgz
! pip install -q findspark
! pip install pyspark

###3. ariable d'environnement PYSPARK_SUBMIT_ARGS

In [57]:
# Set up required environment variables
import os
os.environ["PYSPARK_SUBMIT_ARGS"] = "--packages org.apache.spark:spark-avro_2.11:2.4.5 pyspark-shell"

###4. Création d'un objet SparkContext

In [ ]:
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-8-openjdk-amd64"
os.environ["SPARK_HOME"] = "/content/spark-3.5.0-bin-hadoop3"

import findspark

findspark.init("spark-3.5.0-bin-hadoop3")


from pyspark import SparkContext, SparkConf


configuration = SparkConf().setAppName("name").setMaster("local[4]")
sc = SparkContext(conf=configuration)

###5. Création d'un objet de type SparkSession

In [59]:
from pyspark.sql import SparkSession
# Créer un objet SparkSession
spark = SparkSession.builder.appName("TP_2_Spark").getOrCreate()

##3.Données

### 1. ,2. Téléchargement et décompression des données textuelles à l’adresse suivante :

In [60]:
!wget -q http://qwone.com/~jason/20Newsgroups/20news-19997.tar.gz
!tar xf 20news-19997.tar.gz

###3. b,c Chargement des données alt.atheism et rec.sport.baseball dans deux RDD

In [129]:
rdd1_atheism = sc.wholeTextFiles("20_newsgroups/alt.atheism/*")
rdd2_baseball = sc.wholeTextFiles("20_newsgroups/rec.sport.baseball/*")

###4. Séparer le corps du message de l’entête. (séparation sur “\n\n” ?) (x[0], entete, corps)

In [128]:
rdd1 = rdd1_atheism.map(lambda x: (x[0], x[1].split("\n\n", 1)))
rdd2 = rdd2_baseball.map(lambda x: (x[0], x[1].split("\n\n", 1)))

###5. Extraire quelques champs de l'entête

Procédé pour extraire quelques champs de l'entête : Newsgroups et Organisation

In [127]:
#prenons le premier element du rdd1
a = rdd1.first()
#Prenons le premier élément qui est contituer entre autre des méta données
#print(a[0])
#print(a[1])

#extraction du contenu 49960
# prenons le premier élément et consultons le
#print(a[1][0])

#On parcourt les éléments
b = a[1][0].split('Newsgroups:')[0]
#print(b)
b = a[1][0].split('Newsgroups:')[1]
#print(b)
b = a[1][0].split('Newsgroups:')[1].split("\n")[0]
#print(b)
# ajout d'un test d'existance de Newsgroups dans l'entête
b = a[1][0].split('Newsgroups:')[1].split("\n")[0] if "Newsgroups: " in a[1][0] else None
print('-------------------------------------------------------------')
print(b)
print('-------------------------------------------------------------')

-------------------------------------------------------------
 alt.atheism,alt.atheism.moderated,news.answers,alt.answers
-------------------------------------------------------------


####5. Extraire quelques champs de l'entête s'ils existent sinon None à la place

In [16]:
# Extraire quelques champs de l'entête
rdd11 = rdd1.map(lambda x: (
    x[0],
    x[1][0],
    x[1][0].split('Newsgroups:')[1].split("\n")[0] if "Newsgroups: " in x[1][0] else None,
    x[1][0].split("Organization: ")[1].split("\n")[0] if "Organization: " in x[1][0] else None
))

rdd21 = rdd2.map(lambda x: (
    x[0],
    x[1][0],
    x[1][0].split('Newsgroups:')[1].split("\n")[0] if "Newsgroups: " in x[1][0] else None,
    x[1][0].split("Organization: ")[1].split("\n")[0] if "Organization: " in x[1][0] else None
))

### 6. Fusion des deux RDD (union)

In [122]:
# Fusionner les deux RDD (union)
rdd = rdd21.union(rdd11)
#rdd.take(2)

### 7. Transformer le nouveau RDD obtenu pour que chaque élément soit de type
pyspark.sql.Row

In [120]:
from pyspark.sql import Row
row_rdd = rdd.map(lambda x: Row(file=x[0], header=x[1], newsgroup=x[2],organization=x[3]))

### 8. Créer un objet de type DataFrame à partir du RDD précédent

In [119]:
df = spark.createDataFrame(row_rdd)
#df.printSchema()


###9. Sauvegarder la DataFrame au format Avro

In [118]:
 # Convert the Spark DataFrame back to a Pandas DataFrame using Arrow
result_pdf = df.select("*").toPandas()
#result_pdf.head()

In [35]:
!pip install pandavro

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 9.8 MB/s eta 0:00:00


In [37]:
import pandavro as pdx

pdx.to_avro('data.avro', result_pdf)

In [38]:
!ls -lh

total 400M
-rw-r--r--  1 root  root   17M Sep 22  2014 20news-19997.tar.gz
drwxr-xr-x 22 28757 staff 4.0K Apr  3  1999 20_newsgroups
-rw-r--r--  1 root  root  1.4M Feb  6 09:52 data.avro
drwxr-xr-x  1 root  root  4.0K Feb  2 14:53 sample_data
drwxr-xr-x 13  1000  1000 4.0K Sep  9 02:08 spark-3.5.0-bin-hadoop3
-rw-r--r--  1 root  root  382M Sep  9 02:10 spark-3.5.0-bin-hadoop3.tgz


In [ ]:
filename = 'data.avro'
saved = pdx.read_avro(filename)
print(saved)

### 10. Sauvegarder la DataFrame au format Parquet

In [40]:
!pip install pyarrow

In [41]:
import pyarrow as pa
import pyarrow.parquet as pq

In [44]:
table = pa.Table.from_pandas(result_pdf)
pq.write_table(table, 'data.parquet')

In [45]:
!ls -lh

total 401M
-rw-r--r--  1 root  root   17M Sep 22  2014 20news-19997.tar.gz
drwxr-xr-x 22 28757 staff 4.0K Apr  3  1999 20_newsgroups
-rw-r--r--  1 root  root  1.4M Feb  6 09:52 data.avro
-rw-r--r--  1 root  root  379K Feb  6 09:56 data.parquet
-rw-r--r--  1 root  root  379K Feb  6 09:56 df.parquet
drwxr-xr-x  1 root  root  4.0K Feb  2 14:53 sample_data
drwxr-xr-x 13  1000  1000 4.0K Sep  9 02:08 spark-3.5.0-bin-hadoop3
-rw-r--r--  1 root  root  382M Sep  9 02:10 spark-3.5.0-bin-hadoop3.tgz


In [ ]:
table2 = pq.read_table('data.parquet')
table2.to_pandas()

##4. Analyse descriptive


### 1. Vérifier qu'on a bien deux catégories différentes de documents

In [48]:
df.select("newsgroup").distinct().show()

+--------------------+
|           newsgroup|
+--------------------+
| rec.sport.baseba...|
| rec.sport.baseba...|
| rec.sport.baseba...|
| rec.sport.baseba...|
| rec.sport.baseba...|
| rec.collecting.c...|
|  rec.sport.baseball|
| rec.sport.baseba...|
| soc.culture.cana...|
| rec.sport.baseba...|
| rec.sport.baseba...|
|   rec.sport.base...|
| talk.religion.mi...|
| alt.atheism,rec....|
| alt.atheism,talk...|
| alt.atheism,talk...|
| alt.atheism,alt....|
| alt.atheism,alt....|
| alt.atheism,talk...|
| alt.atheism,talk...|
+--------------------+
only showing top 20 rows



### 2. Donner le nombre d'organisations différentes

In [52]:
df.select("organization").distinct().count()

486

### 3. Statistiques descriptives sur les champs extraits

In [53]:
df.describe().show()

+-------+--------------------+--------------------+--------------------+--------------------+
|summary|                file|              header|           newsgroup|        organization|
+-------+--------------------+--------------------+--------------------+--------------------+
|  count|                2000|                2000|                2000|                1937|
|   mean|                NULL|                NULL|                NULL|                NULL|
| stddev|                NULL|                NULL|                NULL|                NULL|
|    min|file:/content/20_...|From: Alan.Olsen@...|   rec.sport.base...| Wright State Uni...|
|    max|file:/content/20_...|Xref: cantaloupe....| talk.religion.mi...|       worldbank.org|
+-------+--------------------+--------------------+--------------------+--------------------+



## 5. Transformation du texte

### 2. Découper les documents en listes de mots à l'aide de Tokenizer

In [104]:
#Création d'un RDD contenant les contenu des documents
rddtest = rdd1_atheism.map(lambda x: (x[0], x[1].split("\n\n", 1)))

a = rddtest.first()[1][1].split('\n\n\n')
#a

rdda = rdd1.map(lambda x: (
    x[0],
    x[1][1].split('\n\n\n')
))

rddb = rdd2.map(lambda x: (
    x[0],
    x[1][1].split('\n\n\n')
))

rdd_merge = rdda.union(rddb)

#rdd_merge.first()

In [105]:
from pyspark.ml.feature import Tokenizer, HashingTF

row_rdd = rdd_merge.map(lambda x: Row(file=x[0], contenu=x[1][0]))
df = spark.createDataFrame(row_rdd)
#df.printSchema()

tokenizer = Tokenizer(inputCol="contenu", outputCol="words")
df_words = tokenizer.transform(df)
#df_words.show()

### 3. Créer une représentation vectorielle des documents à l'aide de HashingTF

In [106]:
hashingTF = HashingTF(inputCol="words", outputCol="features", numFeatures=20)
df_features = hashingTF.transform(df_words)

#df_features.show()

##6. Grouper les documents qui ont des représentations vectorielles proches

###2. Utiliser l’algorithme KMeans avec un nombre du cluster égal à 2 (pour essayer de retrouver les 2 catégories qu’on a

In [108]:
from pyspark.ml.clustering import KMeans

In [117]:
kmeans = KMeans(k=2, featuresCol="features", predictionCol="cluster")
model = kmeans.fit(df_features)
result = model.transform(df_features)

#Résultats
result.select("file", "contenu", "cluster").show()

+--------------------+--------------------+-------+
|                file|             contenu|cluster|
+--------------------+--------------------+-------+
|file:/content/20_...|Archive-name: ath...|      0|
|file:/content/20_...|Archive-name: ath...|      1|
|file:/content/20_...|In article <65974...|      0|
|file:/content/20_...|dmn@kepler.unh.ed...|      0|
|file:/content/20_...|In article <N4HY....|      0|
|file:/content/20_...|In article <1993A...|      0|
|file:/content/20_...|arromdee@jyusenky...|      0|
|file:/content/20_...|In article <11412...|      0|
|file:/content/20_...|(reference line t...|      0|
|file:/content/20_...|kmr4@po.CWRU.edu ...|      0|
|file:/content/20_...|livesey@solntze.w...|      0|
|file:/content/20_...|sandvik@newton.ap...|      0|
|file:/content/20_...|wpr@atlanta.dg.co...|      0|
|file:/content/20_...|arromdee@jyusenky...|      0|
|file:/content/20_...|bobbe@vice.ICO.TE...|      0|
|file:/content/20_...|bobbe@vice.ICO.TE...|      0|
|file:/conte

## 7. Pour aller plus loin (optionnel)

### 1. Pondérer les mots avec la formule Tf-Idf (avant KMeans)

In [111]:
from pyspark.ml.feature import IDF

idf = IDF(inputCol="features", outputCol="tfidf_features")
idf_model = idf.fit(df_features)
df_tfidf = idf_model.transform(df_features)

### 2. Normaliser les vecteurs représentant les documents (avant KMeans)

In [113]:
from pyspark.ml.feature import Normalizer

normalizer = Normalizer(inputCol="tfidf_features", outputCol="normalized_features", p=2.0)
df_normalized = normalizer.transform(df_tfidf)

# Utiliser ces données normalisées pour le clustering KMeans
kmeans_normalized = KMeans(k=2, featuresCol="normalized_features", predictionCol="cluster_normalized")
model_normalized = kmeans_normalized.fit(df_normalized)
result_normalized = model_normalized.transform(df_normalized)

# Afficher les résultats du clustering avec données normalisées
result_normalized.select("file", "contenu", "cluster_normalized").show()

+--------------------+--------------------+------------------+
|                file|             contenu|cluster_normalized|
+--------------------+--------------------+------------------+
|file:/content/20_...|Archive-name: ath...|                 0|
|file:/content/20_...|Archive-name: ath...|                 1|
|file:/content/20_...|In article <65974...|                 1|
|file:/content/20_...|dmn@kepler.unh.ed...|                 1|
|file:/content/20_...|In article <N4HY....|                 1|
|file:/content/20_...|In article <1993A...|                 1|
|file:/content/20_...|arromdee@jyusenky...|                 1|
|file:/content/20_...|In article <11412...|                 1|
|file:/content/20_...|(reference line t...|                 1|
|file:/content/20_...|kmr4@po.CWRU.edu ...|                 1|
|file:/content/20_...|livesey@solntze.w...|                 1|
|file:/content/20_...|sandvik@newton.ap...|                 1|
|file:/content/20_...|wpr@atlanta.dg.co...|            